setup environment

In [1]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader,dataset


In [37]:
SEED= 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    print("Using GPU")
    torch.cuda.manual_seed(SEED)
    device = torch.device("cuda")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



load the dataset

In [38]:
df= pd.read_csv("C:/Users/Shree/Desktop/AI/MachineLearning/assignment1_dataset.csv")
df.shape
df.head()


,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1


drop columns

In [48]:
data= df.drop(columns=["No"], axis=0)
df.shape

(414, 8)

In [49]:
X= data.drop(columns=["Y house price of unit area"], axis=0)
y= data["Y house price of unit area"].values
df.shape

(414, 8)

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X, df["Y house price of unit area"], test_size=0.2, random_state=SEED)
df.shape

(414, 8)

In [51]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

pytorch function started


In [52]:
class LRdataset(nn.Module):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

#data loader batch size


In [53]:
batch_size = 32
train_dataset = LRdataset(X_train_scaled, y_train)
test_dataset = LRdataset(X_test_scaled, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print(f"Number of training samples: {len(train_dataset)}")


Number of training samples: 331


#5 layer NN
input lenear - relu
4 layer will be for hidden layer [128,128,256,256]
1 output layer


In [59]:
class fivelayerregression(nn.Module):
    def __init__(self, num_features, hiddenlayers):
        super().__init__()
        self.fc1 = nn.Linear(num_features, hiddenlayers[0])
        self.fc2 = nn.Linear(hiddenlayers[0], hiddenlayers[1])
        self.fc3 = nn.Linear(hiddenlayers[1], hiddenlayers[2])
        self.fc4 = nn.Linear(hiddenlayers[2], hiddenlayers[3])
        self.fc5 = nn.Linear(hiddenlayers[3], 1)

        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

hyperparameter

In [ ]:
num_features = X_train_scaled.shape[1]
print(f"Number of features: {num_features}")
hiddenlayers = [128, 128, 256, 256]
model = fivelayerregression(num_features, hiddenlayers)
print(model)



In [68]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {total_params}")

Total number of parameters in the model: 116481


In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs): 
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

start with SGD

In [ ]:
model_sgd = fivelayerregression(num_features, hiddenlayers)
criterion = nn.MSELoss()
print(f"Total number of parameters in the model: {total_params}")
optimizer_sgd = torch.optim.SGD(model_sgd.parameters(), lr=0.01)
print("Training model with SGD optimizer...")
sgd_list = train_model(model_sgd, train_loader, criterion, optimizer_sgd, num_epochs=100)


Training model with SGD optimizer...
Total number of parameters in the model: 116481
